# **Project Name**    - **STRAVA Fitness Application**



##### **Project Type**    - EDA/Regression/Classification/Unsupervised
##### **Contribution**    - Individual
##### **Team Member 1 -** - Sourabh verma
##### **Team Member 2 -**
##### **Team Member 3 -**
##### **Team Member 4 -**

# **Project Summary -**


Based on the initial data exploration, here's what we know about the merged_df dataset:

**Dimensions:** It contains 937 rows and 18 columns after dropping duplicates based on id.

**Data Types:** The dataset includes a mix of int64, float64, and object (for activitydate). The activitydate column was successfully converted to datetime objects and then formatted to 'YYYY-MM-DD' strings.

**Duplicates:** Initially, 6 duplicate entries were identified across all columns before dropping duplicates based on id.

**Missing Values:** The columns totalsleeprecords, totalminutesasleep, and totaltimeinbed have 20 missing values each. These columns are derived from the sleep_df and indicate that not all activity records have corresponding sleep data. The missing values were visually confirmed with ms.bar(merged_df)


# **GitHub Link -**

Provide your GitHub Link here.

# **Problem Statement**


**Problem Statement:**

Strava (or a similar fitness platform) faces the challenge of maximizing user engagement and retention across its diverse user base. Current approaches may not effectively address the varied and fluctuating activity and sleep patterns observed among users. Specifically, highly active users show consistent engagement, while lightly active and sedentary users demonstrate significant drop-offs in activity, especially on certain days. This variability indicates that a single, universal strategy for motivating users is ineffective, potentially leading to suboptimal user experience and churn. The problem is to identify and leverage distinct behavioral trends to develop targeted, data-driven strategies that enhance engagement and foster sustained platform growth.

#### **Define Your Business Objective?**

The primary objective of this analysis is to uncover distinct behavioral trends in how users interact with their smart fitness devices. By identifying patterns in daily activity levels, sleep habits, and device usage, we aim to provide actionable, data-driven recommendations that will guide the marketing team in designing targeted campaigns to improve user engagement, retention, and overall platform growth.

Our exploratory data analysis reveals that users do not interact with their fitness trackers uniformly; their behavior is highly segmented based on their baseline activity levels and the day of the week.

We discovered that while highly active users maintain consistent momentum (peaking on weekends), lightly active and sedentary users experience significant drop-offs in movement during specific days. Furthermore, our correlation analysis highlights a strong relationship between total daily steps and overall calorie expenditure, but reveals a much more complex, non-linear relationship between physical activity and total minutes asleep. Ultimately, the data indicates that a "one-size-fits-all" approach to user motivation is ineffective.

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 20 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import missingno as ms
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import plotly.express as px
import random
from wordcloud import WordCloud
import ast
from PIL import Image
import IPython.display as display
import statsmodels as stat
import geopandas as geo

### Dataset Loading

In [ ]:

# 1. Download the file from GitHub and explicitly name it 'dataset.zip'
!wget -O dataset.zip "https://raw.githubusercontent.com/sourabhverma168-bot/STRAVA/main/dataset.zip"

# 2. Unzip 'dataset.zip' into a folder called 'dataset'
!unzip dataset.zip -d dataset


In [ ]:
import pandas as pd
import glob
import os

# 1. Search for every .csv file in your Colab environment
csv_paths = glob.glob('**/*.csv', recursive=True)

# 2. Create an empty dictionary to hold all our dataframes
fitabase_data = {}

# 3. Loop through, read each one, and store it by its file name
for path in csv_paths:
    file_name = os.path.basename(path)
    # Try-except block just in case there's a weird hidden file
    try:
        fitabase_data[file_name] = pd.read_csv(path)
    except Exception as e:
        print(f"Skipped {file_name}: {e}")

print(f"✅ Successfully loaded {len(fitabase_data)} files into memory!")



In [ ]:
# Load Dataset
activity_df = fitabase_data["dailyActivity_merged.csv"]
dailycalorie_df = fitabase_data["dailyCalories_merged.csv"]
dailyintensity_df = fitabase_data["dailyIntensities_merged.csv"]
dailysteps_df = fitabase_data["dailySteps_merged.csv"]
sleep_df = fitabase_data["sleepDay_merged.csv"]
weightloginfo_df = fitabase_data["weightLogInfo_merged.csv"]

# 2. Function to clean data specifically for PostgreSQL
def clean_for_sql(df):
    # Standardize column names: lowercase, replace spaces/special chars
    df.columns = df.columns.str.strip().str.lower().str.replace(r'[^a-z0-9_]', '', regex=True)

    # Remove commas and single quotes from all string columns
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype(str).str.replace(',', '', regex=False).str.replace("'", "", regex=False)
    return df

activity_df = clean_for_sql(activity_df)
sleep_df = clean_for_sql(sleep_df)

# 3. Format dates to strictly YYYY-MM-DD
activity_df['activitydate'] = pd.to_datetime(activity_df['activitydate']).dt.strftime('%Y-%m-%d')
sleep_df['sleepday'] = pd.to_datetime(sleep_df['sleepday']).dt.strftime('%Y-%m-%d')

# 4. Merge Activity and Sleep on User ID and Date
merged_df = pd.merge(activity_df, sleep_df, left_on=['id', 'activitydate'], right_on=['id', 'sleepday'], how='left')

# Drop the redundant 'sleepday' column after merging
if 'sleepday' in merged_df.columns:
    merged_df.drop(columns=['sleepday'], inplace=True)



In [ ]:
# 1. Check the starting data to see if the missing days were already lost
print(f"Starting Activity Data Shape: {activity_df.shape}")
print(f"Starting Sleep Data Shape: {sleep_df.shape}")

# 2. Function to clean data specifically for PostgreSQL
def clean_for_sql(df):
    # Standardize column names: lowercase, replace spaces/special chars
    df.columns = df.columns.str.strip().str.lower().str.replace(r'[^a-z0-9_]', '', regex=True)

    # Remove commas and single quotes from all string columns
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype(str).str.replace(',', '', regex=False).str.replace("'", "", regex=False)
    return df

# Apply the cleaning function
activity_clean = clean_for_sql(activity_df.copy())
sleep_clean = clean_for_sql(sleep_df.copy())

# 3. Format dates safely
# Using format='mixed' helps if the two XLSX files for your Strava project exported dates in varying formats
activity_clean['activitydate'] = pd.to_datetime(activity_clean['activitydate'], format='mixed', errors='coerce').dt.strftime('%Y-%m-%d')
sleep_clean['sleepday'] = pd.to_datetime(sleep_clean['sleepday'], format='mixed', errors='coerce').dt.strftime('%Y-%m-%d')

# 4. Merge Activity and Sleep
merged_df = pd.merge(
    activity_clean,
    sleep_clean,
    left_on=['id', 'activitydate'],
    right_on=['id', 'sleepday'],
    how='left'
)

# 5. Drop redundant column
if 'sleepday' in merged_df.columns:
    merged_df.drop(columns=['sleepday'], inplace=True)

print(f"\nFinal Merged Shape: {merged_df.shape}")
print(f"Unique days remaining: {merged_df['activitydate'].nunique()}")

### Dataset First View

In [ ]:
# Dataset First Look
merged_df.head()

In [ ]:
merged_df.tail()

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
merged_df.shape

### Dataset Information

In [ ]:
# Dataset Info
merged_df.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
duplicates = merged_df.duplicated(keep=False)

# Count the duplicate values
duplicate_count = duplicates.value_counts()

print(duplicate_count)

In [ ]:
merged_df.drop_duplicates(["id"],keep='first',inplace=True)

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
merged_df.isnull().sum()

In [ ]:
# Visualizing the missing values
ms.bar(merged_df)


### What did you know about your dataset?


Based on the initial data exploration, here's what we know about the merged_df dataset:

**Dimensions**: It contains 937 rows and 18 columns after dropping duplicates based on id.

**Data Types**: The dataset includes a mix of int64, float64, and object (for activitydate). The activitydate column was successfully converted to datetime objects and then formatted to 'YYYY-MM-DD' strings.

**Duplicates**: Initially, 6 duplicate entries were identified across all columns before dropping duplicates based on id.

**Missing Values**: The columns totalsleeprecords, totalminutesasleep, and totaltimeinbed have 20 missing values each. These columns are derived from the sleep_df and indicate that not all activity records have corresponding sleep data. The missing values were visually confirmed with ms.bar(merged_df)

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
merged_df.columns

In [ ]:
# Dataset Describe
merged_df.describe()

### Variables Description

Answer Here

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
for column in merged_df.columns:
    print(f"Column '{column}': {merged_df[column].nunique()} unique values")

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.
# ==========================================
# DATA WRANGLING & FEATURE ENGINEERING
# ==========================================


# 1. Drop duplicate rows (The sleep dataset is notorious for having a few duplicates)
merged_df.drop_duplicates(inplace=True)
print(f"Duplicates removed! Cleaned dataset shape: {merged_df.shape}")

# 2. Convert the 'activitydate' back to a datetime object so we can extract the day of the week
merged_df['activitydate'] = pd.to_datetime(merged_df['activitydate'])

# 3. Create a 'Day_of_Week' column (e.g., Monday, Tuesday)
merged_df['day_of_week'] = merged_df['activitydate'].dt.day_name()

# 4. Create an 'Activity_Level' category based on total steps
# (Under 5k = Sedentary, 5k-10k = Lightly Active, 10k+ = Very Active)
def categorize_activity(steps):
    if steps < 5000:
        return 'Sedentary'
    elif steps >= 5000 and steps <= 10000:
        return 'Lightly Active'
    else:
        return 'Very Active'

merged_df['activity_level'] = merged_df['totalsteps'].apply(categorize_activity)

# 5. Fill missing sleep values with 0 (since not everyone logged sleep every day)
merged_df['totalminutesasleep'] = merged_df['totalminutesasleep'].fillna(0)
merged_df['totaltimeinbed'] = merged_df['totaltimeinbed'].fillna(0)

print("\n--- New Columns Added ---")

display.display(merged_df[['activitydate', 'day_of_week', 'totalsteps', 'activity_level', 'totalminutesasleep']].head())

In [ ]:
# This will count every single day in the entire dataset, not just the top 5 rows
print(merged_df['day_of_week'].value_counts())

### What all manipulations have you done and insights you found?

Now we have updated the notebook with a summary of the data manipulations and initial insights. These include removing duplicates, engineering the day of the week, categorizing activity levels by step count, and handling missing sleep data.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart - 1 visualization code

# Chart - 1 visualization code
plt.figure(figsize=(10, 6))

# Plotting a histogram with a Kernel Density Estimate (KDE) curve
sns.histplot(merged_df['totalsteps'], bins=20, kde=True, color='skyblue')

# Setting labels and title
plt.title('Distribution of Total Daily Steps', fontsize=14, fontweight='bold')
plt.xlabel('Total Steps', fontsize=12)
plt.ylabel('Frequency (Days)', fontsize=12)

# Grid and layout
plt.grid(axis='y', alpha=0.5)
plt.tight_layout()

# Save and show
plt.savefig('distribution_total_steps.png')
plt.show()

##### 1. Why did you pick the specific chart?

A histogram with a density curve is the standard visualization for understanding the distribution, spread, and central tendency of a single continuous variable, instantly highlighting common ranges and extreme outliers.

##### 2. What is/are the insight(s) found from the chart?

The distribution of daily steps is right-skewed, revealing that the vast majority of logged days fall between 0 and 15,000 steps (peaking around 10,000), with a long tail indicating occasional, highly active outlier days reaching up to 35,000 steps.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Since most users naturally plateau around the 10,000 to 12,000 step mark, the platform should keep default daily goals within this realistic, achievable threshold to prevent burnout, while introducing rare "Milestone Badges" to uniquely reward the outliers who hit 20k+ steps.

#### Chart - 2

In [ ]:
# Chart - 2 visualization code
# 1. Count how many times each activity level appears in your data
activity_counts = merged_df['activity_level'].value_counts()

# 2. Set the size of the chart
plt.figure(figsize=(8, 8))

# 3. Create the pie chart using Matplotlib
plt.pie(
    activity_counts,

    labels=activity_counts.index,
    autopct='%1.1f%%', # This formats the numbers as percentages (e.g., 45.2%)
    startangle=140,    # Rotates the starting point of the pie slightly
    colors=sns.color_palette('pastel') # Gives the slices soft, pastel colors
)

# 4. Add a title
plt.title('Percentage of Logs by Activity Level', fontsize=16)

# 5. Show the plot
plt.show()

##### 1. Why did you pick the specific chart?

Pie charts are the standard go-to for showing how parts make up a whole.

##### 2. What is/are the insight(s) found from the chart?

 This will show us the distribution of how active the users were across all their logged days.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 3

In [ ]:
# Chart - 3 visualization code

# 1. Set the order of the days so the chart reads naturally
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# 2. Set the size of the chart
plt.figure(figsize=(14, 6))

# 3. Create the bar chart using Seaborn
# errorbar=None removes the little error lines on top of the bars to keep it clean
sns.barplot(
    data=merged_df,
    x='day_of_week',
    y='totalsteps', hue = 'totalsteps',
    order=day_order,
    errorbar=None,
    palette='viridis' # A nice, built-in color palette
)

# 4. Add titles and labels
plt.title('Average Total Steps by Day of the Week', fontsize=16)
plt.xlabel('Day of the Week', fontsize=12)
plt.ylabel('Average Steps', fontsize=12)

# 5. Show the plot
plt.show()



##### 1. Why did you pick the specific chart?

It allows us to compare the volume of different activity levels side-by-side across each day of the week to spot behavioral patterns.

##### 2. What is/are the insight(s) found from the chart?

It reveals which specific days users are most likely to be highly active versus when they tend to be the most sedentary.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Fitness tracking platforms can use this to send targeted, timely push notifications—like prompting a workout on a typically sedentary Thursday—to boost user engagement and overall health.

#### Chart - 4

In [ ]:
# Chart - 4 visualization code
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Set the size of your chart so it's wide enough for clustered bars
plt.figure(figsize=(14, 7))

# 2. Define the logical order of the days so it reads cleanly
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# 3. Create the clustered bar chart using Seaborn's barplot
# The 'hue' parameter is the secret ingredient that creates the clusters!
sns.barplot(
    data=merged_df,
    x='day_of_week',
    y='totalsteps',
    hue='activity_level',
    order=day_order,
    errorbar=None, # Keeps the chart clean by removing error bars
    palette='Set2' # A colorblind-friendly palette
)

# 4. Add clear titles and labels for your EDA presentation
plt.title('Average Total Steps by Day of the Week, Clustered by Activity Level', fontsize=16)
plt.xlabel('Day of the Week', fontsize=12)
plt.ylabel('Average Total Steps', fontsize=12)

# 5. Position the legend so it doesn't block your data
plt.legend(title='Activity Level', loc='upper right')

# 6. Display the chart
plt.show()

##### 1. Why did you pick the specific chart?

A clustered bar chart perfectly enables a multi-dimensional comparison, allowing us to simultaneously evaluate average steps across days of the week while segmenting by user activity levels.

##### 2. What is/are the insight(s) found from the chart?

"Very Active" users peak in their step count on Saturdays (approaching 15,000 steps), whereas "Lightly Active" users actually experience a slight decline in movement during the weekend.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This presents a prime opportunity to design targeted push notifications—such as sending weekend walking challenges to motivate lightly active users, while offering rewards or recovery tips to highly active users on Sundays.

#### Chart - 5

In [ ]:
# Chart - 5 visualization code
plt.figure(figsize=(10, 6))

# Create the scatter plot
# 'alpha' controls the transparency of the dots
sns.scatterplot(
    data=merged_df,
    x='totalsteps',
    y='totalminutesasleep',
    alpha=0.8,
    color='red',
    edgecolor=None
)

# Add titles and labels
plt.title('Relationship Between Total Steps and Minutes Asleep', fontsize=16)
plt.xlabel('Total Steps', fontsize=12)
plt.ylabel('Total Minutes Asleep', fontsize=12)

# Display the chart
plt.show()

##### 1. Why did you pick the specific chart?

A scatter plot is the most effective visualization for identifying potential correlations, trends, or outliers between two continuous numerical variables.

##### 2. What is/are the insight(s) found from the chart?

This visual reveals whether taking more steps throughout the day directly translates to more minutes asleep, while also highlighting where the vast majority of user habits naturally cluster (likely around the middle).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

By identifying the "sweet spot" of steps that leads to the best sleep, health platforms can create personalized daily step goals that explicitly promise users a better night's rest, boosting both app retention and user health.

#### Chart - 6

In [ ]:
# Chart - 6 visualization code
plt.figure(figsize=(10, 6))

# sns.regplot creates both the scatter points AND the trendline
sns.regplot(
    data=merged_df,
    x='totalsteps',
    y='totalminutesasleep',
    scatter_kws={'alpha': 0.5, 'color': 'royalblue'}, # Styles the dots (making them transparent)
    line_kws={'color': 'red', 'linewidth': 2}        # Styles the trendline so it stands out
)

# Add titles and labels
plt.title('Correlation Between Total Steps and Minutes Asleep', fontsize=16)
plt.xlabel('Total Steps', fontsize=12)
plt.ylabel('Total Minutes Asleep', fontsize=12)

# Display the chart
plt.show()

##### 1. Why did you pick the specific chart?

A scatter plot paired with a regression line (trendline) perfectly visualizes the distribution between two continuous variables, instantly revealing the mathematical trajectory and strength of their correlatio

##### 2. What is/are the insight(s) found from the chart?

he upward slope of the red trendline shows a slight positive correlation, meaning users who take more steps generally get slightly more sleep. However, the wide spread of the dots and the dense cluster at 0 minutes indicate that many users simply forget to track their sleep, making steps alone an imperfect predictor of sleep duration.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Marketing teams can leverage the positive trendline to run motivational campaigns like "Move more to sleep better!", while the product team should trigger evening push notifications reminding users to wear their devices to bed to fix the high volume of missing sleep logs.

#### Chart - 7

In [ ]:
# Chart - 7 visualization code
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Turn on dark mode for this specific plot
plt.style.use('dark_background')

# 2. List the columns you want to visualize
# (Make sure these perfectly match the columns in your merged_df)
cols_to_plot = [
    'veryactiveminutes', 'fairlyactiveminutes', 'lightlyactiveminutes',
    'veryactivedistance', 'totalsteps', 'totaldistance',
    'trackerdistance'
]

# 3. Set up the 3x3 grid
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
axes = axes.flatten() # Flattens the 3x3 grid into a simple list so we can loop through it

# 4. Loop through each column and create a histogram
for i, col in enumerate(cols_to_plot):
    if col in merged_df.columns:
        # Create the histogram with the KDE line, using a nice "Fitbit Green"
        sns.histplot(data=merged_df, x=col, kde=True, color='violet', ax=axes[i])
        axes[i].set_title(f'Distribution of {col}', fontsize=12)
        axes[i].set_ylabel('Count')

# 5. Hide any extra, empty subplots (since we have 7 columns but 9 slots)
for j in range(len(cols_to_plot), len(axes)):
    fig.delaxes(axes[j])

# 6. Adjust spacing and show
plt.tight_layout()
plt.show()

# 7. Reset back to the default white background for your future charts
plt.style.use('default')

##### 1. Why did you pick the specific chart?

The above chart is better for comparing numerical data types with each other and itself.

##### 2. What is/are the insight(s) found from the chart?

"Visualizing the distributions reveals that most activity metrics are heavily right-skewed, indicating that while most users log baseline activity, a small group of highly active 'power users' pushes the maximum values significantly higher."

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

From above insights we can conclude that the strava app monitor on these parameters as they are well achieved by the user but surely there is hope for improvement.

#### Chart - 8

In [ ]:
# Chart - 8 visualization code
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Group the data by date to find the average steps for each specific day
daily_avg = merged_df.groupby('activitydate')['totalsteps'].mean().reset_index()

# 2. Set the size of the chart
plt.figure(figsize=(14, 6))

# 3. Create the line chart
# 'marker='o'' adds a visible dot for every single day on the line
sns.lineplot(
    data=daily_avg,
    x='activitydate',
    y='totalsteps',
    marker='o',
    color='mediumseagreen',
    linewidth=2.5
)

# 4. Format the x-axis so the dates don't overlap and look messy
plt.xticks(rotation=45)

# 5. Add titles, labels, and a subtle grid for readability
plt.title('Trend of Average Total Steps Over Time', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Average Total Steps', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)

# 6. Display the chart cleanly
plt.tight_layout() # Ensures the rotated dates don't get cut off at the bottom
plt.show()

##### 1. Why did you pick the specific chart?

A time-series line chart is the optimal visualization for tracking continuous data over a chronological period, allowing us to spot long-term trends, momentum shifts, or recurring seasonal spikes.

##### 2. What is/are the insight(s) found from the chart?

This visualization instantly reveals the "momentum curve" of the users—showing us exactly when average activity peaks during the month and pinpointing the exact dates when user motivation begins to dip

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

By identifying the specific time of the month when activity historically drops off, app developers can preemptively launch "Mid-Month Walking Challenges" or send automated, timed push notifications to re-engage users right before they normally quit.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Filter the dataset to ONLY include numerical columns
# This prevents errors when Pandas tries to do math on words like "Monday"
numerical_df = merged_df.select_dtypes(include=['float64', 'int64', 'int32', 'float32'])

# 2. Calculate the correlation matrix
corr_matrix = numerical_df.corr()

# 3. Set up the matplotlib figure size (make it large so it's readable)
plt.figure(figsize=(12, 10))

# 4. Generate the heatmap
sns.heatmap(
    corr_matrix,
    annot=True,          # Prints the exact correlation numbers inside the boxes
    cmap='coolwarm',     # A great color palette: Red = Positive, Blue = Negative correlation
    fmt=".2f",           # Rounds the numbers to 2 decimal places so it looks clean
    linewidths=0.5,      # Adds subtle lines between the boxes
    vmin=-1, vmax=1      # Locks the color scale from -1 to 1
)

# 5. Add a title and format
plt.title('Correlation Heatmap of Fitness Metrics', fontsize=18, pad=20)
plt.xticks(rotation=45, ha='right') # Angles the bottom labels so they don't overlap

# 6. Display the chart
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

A correlation heatmap allows us to mathematically validate our hypotheses by calculating and color-coding the exact relationship strength between every single numerical variable simultaneously.

##### 2. What is/are the insight(s) found from the chart?

"The heatmap reveals a very strong positive correlation between totalsteps and calories, while also showing a surprising lack of correlation between sedentaryminutes and totalminutesasleep."

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code

import matplotlib.pyplot as plt
import seaborn as sns

# 1. Select a "highlight reel" of key numerical variables
# We keep this to 4 metrics so the final grid is large and readable
cols_to_plot = ['totalsteps', 'totaldistance', 'calories', 'totalminutesasleep']

# 2. Filter the dataframe to just these columns, PLUS your categorical column for color!
pairplot_df = merged_df[cols_to_plot + ['activity_level']].dropna()

# 3. Create the pairplot
# 'hue' color-codes the dots based on how active the user is
# 'corner=True' is a pro-trick: it removes the redundant charts on the top right half of the grid
sns.pairplot(
    data=pairplot_df,
    hue='activity_level',
    palette='Set1',
    corner=True,
    plot_kws={'alpha': 0.6} # Makes the overlapping dots slightly transparent
)

# 4. Add a main title to the very top of the grid
plt.suptitle('Pairplot of Key Fitness Metrics by Activity Level', y=1.02, fontsize=16)

# 5. Display the chart
plt.show()

##### 1. Why did you pick the specific chart?

A pairplot is the ultimate high-level overview tool, generating a comprehensive grid of scatter plots and distributions to instantly visualize the relationships, trends, and outliers between multiple key variables simultaneously.

##### 2. What is/are the insight(s) found from the chart?

This visual clearly maps out behavioral clusters—for example, showing that while steps and distance share a perfect linear trajectory, the relationship between calories burned and total minutes asleep is much more scattered across different activity levels.

## **5. Solution to Business Objective**

#### What do you suggest the client to achieve Business Objective ?
Explain Briefly.

To capitalize on these insights, the business should implement the following strategic initiatives:

* **Implement Dynamic Push Notifications:** Utilize the day-of-the-week insights to send targeted alerts. For example, send "Mid-Week Push" notifications to lightly active users on Thursdays when their step counts historically dip, while offering active users "Rest and Recovery" tips on Sundays.

* **Develop Personalized Step-to-Sleep Goals:** Since the scatter plot revealed a specific "sweet spot" where steps correlate with optimal sleep, the app should calculate a personalized daily step goal for each user that explicitly promises to improve their sleep quality.

* **Launch Weekend Gamification:** Capitalize on the Saturday activity spike by introducing weekend-specific community challenges (e.g., "Saturday 10k Step Challenge") to encourage sedentary users to mimic the habits of highly active users.

* **Create In-App User Personas:** Transition from generic dashboards to persona-based interfaces. A user identified as "Sedentary" should see a UI focused on small, achievable hourly movement goals, while "Very Active" users should see advanced metrics like distance trajectories and calorie burns.

# **Conclusion**

The primary objective of this analysis is to uncover distinct behavioral trends in how users interact with their smart fitness devices. By identifying patterns in daily activity levels, sleep habits, and device usage, we aim to provide actionable, data-driven recommendations that will guide the marketing team in designing targeted campaigns to improve user engagement, retention, and overall platform growth.

Our exploratory data analysis reveals that users do not interact with their fitness trackers uniformly; their behavior is highly segmented based on their baseline activity levels and the day of the week.

We discovered that while highly active users maintain consistent momentum (peaking on weekends), lightly active and sedentary users experience significant drop-offs in movement during specific days. Furthermore, our correlation analysis highlights a strong relationship between total daily steps and overall calorie expenditure, but reveals a much more complex, non-linear relationship between physical activity and total minutes asleep. Ultimately, the data indicates that a "one-size-fits-all" approach to user motivation is ineffective.

### ***Hurrah! You have successfully completed your EDA Capstone Project !!!***

In [ ]:
# 1. Force strict MySQL Date format (YYYY-MM-DD)
merged_df['activitydate'] = pd.to_datetime(merged_df['activitydate']).dt.strftime('%Y-%m-%d')

# 2. Fill any lingering empty cells with 0 just to be safe
merged_df = merged_df.fillna(0)

# 3. Export a brand new CSV
file_name = 'mysql_ready_fitness_data.csv'
merged_df.to_csv(file_name, index=False)
print(f"Ready for MySQL! Saved as: {file_name}")

In [ ]:
from google.colab import files
files.download('mysql_ready_fitness_data.csv')

In [ ]:
activity_df